# Agentick Benchmark Dashboard

Unified analysis of **all** benchmark results: PPO (pixel-RL), local LLMs, VLMs, API agents, and baselines.

### Sections
1. **Data Loading** — auto-discover all result directories
2. **Global Overview** — leaderboard, summary table
3. **Cross-Agent Comparison** — PPO vs LLM vs VLM heatmap, radar overlay
4. **Difficulty Scaling** — all agents on one difficulty curve
5. **LLM/VLM Deep-Dive** — harness comparison, obs mode comparison, model size scaling
6. **Per-Task Drill-Down** — heatmaps, sorted bar charts
7. **Cost & Efficiency** — tokens, latency, episode length
8. **Failure Analysis** — zero-success and errored tasks

In [ ]:
import json
import re
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
    "figure.facecolor": "white",
})

DIFF_ORDER = ["easy", "medium", "hard", "expert"]
CATEGORY_ORDER = [
    "navigation", "memory", "reasoning", "skill", "control",
    "combinatorial", "adversarial", "meta", "multi_agent", "compositional",
]
DIFF_COLORS = {"easy": "#2ca02c", "medium": "#ff7f0e", "hard": "#d62728", "expert": "#7f7f7f"}
CAT_CMAP = plt.cm.tab10(np.linspace(0, 1, 10))
CAT_COLOR_MAP = {c: CAT_CMAP[i] for i, c in enumerate(CATEGORY_ORDER)}

## 1. Data Loading

Auto-discovers result directories under `results/` and merges into a single DataFrame.

In [ ]:
RESULTS_BASE = Path("results")
RESULTS_SUBDIRS = [
    "ppo_benchmarks", "api_benchmarks", "llm_benchmarks",
    "vlm_benchmarks", "full_benchmark", "hf_benchmarks",
]

def _parse_agent_type(config_name: str) -> str:
    """Infer agent type from config name."""
    if config_name.startswith("ppo"):
        return "ppo"
    if "vl" in config_name and ("vl4b" in config_name or "vl8b" in config_name):
        return "vlm"
    if any(config_name.startswith(p) for p in ("qwen", "llama", "mistral")):
        return "llm"
    if any(config_name.startswith(p) for p in ("claude", "gpt", "anthropic", "openai")):
        return "api"
    if config_name.startswith("random"):
        return "baseline"
    if config_name.startswith("oracle"):
        return "baseline"
    return "other"

def _parse_config_facets(config_name: str) -> dict:
    """Extract model, harness, obs_mode from config name for LLM/VLM configs."""
    facets = {"model": config_name, "harness": "", "obs_mode": ""}
    # Pattern: qwen3_4b_lang_markov, qwen3_30b_a3b_ascii_reasoner, qwen3_vl4b_lang_markov, etc.
    m = re.match(r"(qwen3_(?:vl)?(?:4b|8b|30b_a3b))_(lang|ascii)_(markov|nonmarkov|reasoner)", config_name)
    if m:
        facets["model"] = m.group(1)
        facets["obs_mode"] = m.group(2)
        facets["harness"] = m.group(3)
    return facets

def load_run_dir(run_dir: Path, config_name: str, rows: list, training_curves: dict):
    """Load results from a single run directory."""
    loaded_keys = set()

    # Primary: per_task/ directory (SLURM per-task jobs)
    per_task_dir = run_dir / "per_task"
    if per_task_dir.exists():
        for task_dir in per_task_dir.iterdir():
            if not task_dir.is_dir():
                continue
            # Handle nested difficulty dirs or flat metrics.json
            for sub in task_dir.iterdir():
                if sub.is_dir():
                    metrics_path = sub / "metrics.json"
                    if metrics_path.exists():
                        with open(metrics_path) as f:
                            m = json.load(f)
                        key = (m.get("task", task_dir.name), m.get("difficulty", sub.name))
                        if key in loaded_keys:
                            continue
                        loaded_keys.add(key)
                        rows.append({
                            "config": config_name,
                            "agent_type": _parse_agent_type(config_name),
                            **_parse_config_facets(config_name),
                            "run_dir": str(run_dir),
                            "task": m.get("task", task_dir.name),
                            "difficulty": m.get("difficulty", sub.name),
                            "category": m.get("category", "unknown"),
                            "success_rate": m.get("success_rate", 0.0),
                            "mean_return": m.get("mean_return", 0.0),
                            "std_return": m.get("std_return", 0.0),
                            "mean_length": m.get("mean_length", 0.0),
                            "mean_latency": m.get("mean_latency", 0.0),
                            "total_tokens": m.get("total_tokens", 0),
                            "error": m.get("error"),
                        })
                        curve = m.get("training_curve", [])
                        if curve:
                            training_curves[(config_name, key[0], key[1])] = curve
                elif sub.name == "metrics.json":
                    # Flat per-task metrics (from ExperimentRunner)
                    with open(sub) as f:
                        task_data = json.load(f)
                    for diff, diff_data in task_data.get("per_difficulty", {}).items():
                        key = (task_data.get("task_name", task_dir.name), diff)
                        if key in loaded_keys:
                            continue
                        loaded_keys.add(key)
                        metrics = diff_data.get("metrics", {})
                        rows.append({
                            "config": config_name,
                            "agent_type": _parse_agent_type(config_name),
                            **_parse_config_facets(config_name),
                            "run_dir": str(run_dir),
                            "task": key[0],
                            "difficulty": diff,
                            "category": "unknown",
                            "success_rate": metrics.get("success_rate", 0.0),
                            "mean_return": metrics.get("mean_return", 0.0),
                            "std_return": metrics.get("std_return", 0.0),
                            "mean_length": metrics.get("mean_length", 0.0),
                            "mean_latency": metrics.get("mean_latency", 0.0),
                            "total_tokens": metrics.get("total_tokens", 0),
                            "error": None,
                        })

    # Fallback: training_summary.json
    ts_path = run_dir / "training_summary.json"
    if ts_path.exists():
        with open(ts_path) as f:
            ts = json.load(f)
        for key_name, result in ts.get("results", {}).items():
            key = (result["task"], result["difficulty"])
            if key in loaded_keys:
                continue
            loaded_keys.add(key)
            rows.append({
                "config": config_name,
                "agent_type": _parse_agent_type(config_name),
                **_parse_config_facets(config_name),
                "run_dir": str(run_dir),
                "task": result["task"],
                "difficulty": result["difficulty"],
                "category": result.get("category", "unknown"),
                "success_rate": result.get("success_rate", 0.0),
                "mean_return": result.get("mean_return", 0.0),
                "std_return": result.get("std_return", 0.0),
                "mean_length": result.get("mean_length", 0.0),
                "mean_latency": result.get("mean_latency", 0.0),
                "total_tokens": result.get("total_tokens", 0),
                "error": result.get("error"),
            })
            curve = result.get("training_curve", [])
            if curve:
                training_curves[(config_name, result["task"], result["difficulty"])] = curve

# --- Scan ---
rows = []
training_curves = {}

for subdir_name in RESULTS_SUBDIRS:
    subdir = RESULTS_BASE / subdir_name
    if not subdir.exists():
        continue
    for run_dir in sorted(subdir.iterdir()):
        if not run_dir.is_dir():
            continue
        config_name = run_dir.name.rsplit("_2026", 1)[0].rsplit("_2025", 1)[0]
        load_run_dir(run_dir, config_name, rows, training_curves)

# Also scan flat dirs
for run_dir in sorted(RESULTS_BASE.iterdir()):
    if not run_dir.is_dir() or run_dir.name in RESULTS_SUBDIRS:
        continue
    if (run_dir / "per_task").exists() or (run_dir / "training_summary.json").exists():
        config_name = run_dir.name.rsplit("_2026", 1)[0].rsplit("_2025", 1)[0]
        load_run_dir(run_dir, config_name, rows, training_curves)

df = pd.DataFrame(rows)

if df.empty:
    print("No results found! Check that RESULTS_BASE points to the right directory.")
else:
    df = df.drop_duplicates(subset=["config", "task", "difficulty"], keep="last").reset_index(drop=True)
    # Fill unknown categories from training_runner mapping
    try:
        from agentick.experiments.training_runner import TASK_CATEGORIES
        df["category"] = df.apply(
            lambda r: TASK_CATEGORIES.get(r["task"], r["category"]) if r["category"] == "unknown" else r["category"],
            axis=1,
        )
    except ImportError:
        pass

    configs = sorted(df["config"].unique())
    print(f"Loaded {len(df)} rows across {len(configs)} configs")
    print(f"Tasks: {df['task'].nunique()}, Difficulties: {sorted(df['difficulty'].unique())}")
    print(f"Agent types: {dict(df.groupby('agent_type')['config'].nunique())}")
    print(f"\nConfigs:")
    for c in configs:
        n = df[df['config'] == c]['task'].nunique()
        at = df[df['config'] == c]['agent_type'].iloc[0]
        print(f"  [{at:>5s}] {c} ({n} tasks)")

## 2. Global Leaderboard

Ranked by overall success rate (mean across all tasks and difficulties).

In [ ]:
if not df.empty:
    # Build summary table
    summary_rows = []
    for config in configs:
        sub = df[df["config"] == config]
        row = {
            "Config": config,
            "Type": sub["agent_type"].iloc[0],
            "Tasks": sub["task"].nunique(),
        }
        for diff in DIFF_ORDER:
            d = sub[sub["difficulty"] == diff]
            row[f"SR_{diff}"] = f"{d['success_rate'].mean():.1%}" if len(d) > 0 else "—"
        row["SR_overall"] = f"{sub['success_rate'].mean():.1%}"
        row["Ret_overall"] = f"{sub['mean_return'].mean():.3f}"
        row["Solved_easy"] = int((sub[sub['difficulty'] == 'easy']['success_rate'] > 0).sum())
        summary_rows.append(row)

    leaderboard = pd.DataFrame(summary_rows).sort_values(
        "SR_overall", ascending=False, key=lambda s: s.str.rstrip("%").astype(float)
    )
    display(leaderboard.reset_index(drop=True).style.set_caption("Leaderboard (sorted by overall SR)"))

## 3. Cross-Agent Comparison

### 3a. Task x Agent Heatmap (all agents, mean success rate across difficulties)

In [ ]:
if not df.empty and len(configs) >= 2:
    agent_task = df.groupby(["config", "task"])["success_rate"].mean().unstack(level=0)
    task_cat = df.drop_duplicates("task").set_index("task")["category"]
    cat_rank = {c: i for i, c in enumerate(CATEGORY_ORDER)}
    tasks_sorted = sorted(agent_task.index, key=lambda t: (cat_rank.get(task_cat.get(t, "unknown"), 99), t))
    agent_task = agent_task.reindex(index=tasks_sorted)

    fig, ax = plt.subplots(figsize=(max(8, len(configs) * 0.9), max(8, len(tasks_sorted) * 0.28)))
    im = ax.imshow(agent_task.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(agent_task.columns)))
    ax.set_xticklabels(agent_task.columns, rotation=55, ha="right", fontsize=7)
    ax.set_yticks(range(len(tasks_sorted)))
    ylabels = ax.set_yticklabels(tasks_sorted, fontsize=7)
    for label, task in zip(ylabels, tasks_sorted):
        cat = task_cat.get(task, "unknown")
        label.set_color(CAT_COLOR_MAP.get(cat, "gray"))
    for i in range(len(tasks_sorted)):
        for j in range(len(agent_task.columns)):
            val = agent_task.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=5,
                        color="white" if val < 0.5 else "black")
    plt.colorbar(im, ax=ax, shrink=0.4, label="Success Rate")
    ax.set_title("Cross-Agent Comparison: Task x Agent", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 configs for cross-agent heatmap.")

### 3b. Capability Radar Overlay (per-category success rate, all agents)

In [ ]:
if not df.empty and len(configs) >= 2:
    cat_scores = df.groupby(["config", "category"])["success_rate"].mean().unstack(level=1)
    cat_scores = cat_scores.reindex(columns=CATEGORY_ORDER).fillna(0)
    active_cats = [c for c in CATEGORY_ORDER if cat_scores[c].max() > 0 or cat_scores[c].notna().any()]
    if len(active_cats) >= 3:
        angles = np.linspace(0, 2 * np.pi, len(active_cats), endpoint=False).tolist()
        angles += angles[:1]
        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
        palette = plt.cm.Set2(np.linspace(0, 1, max(len(configs), 2)))
        for idx, config in enumerate(configs):
            values = cat_scores.loc[config, active_cats].tolist() + [cat_scores.loc[config, active_cats[0]]]
            ax.plot(angles, values, "o-", linewidth=1.5, label=config, color=palette[idx], markersize=3)
            ax.fill(angles, values, alpha=0.06, color=palette[idx])
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([c.replace("_", " ").title() for c in active_cats], fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=7)
        ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=6)
        ax.set_title("Capability Radar: All Agents", fontsize=12, y=1.08)
        plt.tight_layout()
        plt.show()
else:
    print("Need at least 2 configs.")

## 4. Difficulty Scaling — All Agents

In [ ]:
if not df.empty:
    diff_agg = df.groupby(["config", "difficulty"]).agg(
        sr_mean=("success_rate", "mean"), sr_std=("success_rate", "std"),
    ).reset_index()

    fig, ax = plt.subplots(figsize=(9, 5))
    palette = plt.cm.tab20(np.linspace(0, 1, max(len(configs), 2)))
    for idx, config in enumerate(configs):
        sub = diff_agg[diff_agg["config"] == config].set_index("difficulty").reindex(DIFF_ORDER)
        x = range(len(DIFF_ORDER))
        ax.plot(x, sub["sr_mean"], "o-", label=config, linewidth=1.5, markersize=4, color=palette[idx])
        ax.fill_between(x, (sub["sr_mean"] - sub["sr_std"]).clip(0),
                        (sub["sr_mean"] + sub["sr_std"]).clip(0, 1), alpha=0.08, color=palette[idx])
    ax.set_xticks(range(len(DIFF_ORDER)))
    ax.set_xticklabels(DIFF_ORDER)
    ax.set_ylabel("Mean Success Rate")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=6, ncol=2, loc="upper right")
    ax.set_title("Difficulty Scaling (all agents)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. LLM/VLM Deep-Dive

Breakdowns by **harness** (markov / nonmarkov / reasoner), **observation mode** (language / ascii), and **model size** (4B / 8B).

In [ ]:
# Filter to LLM/VLM configs that have harness facets
hf_df = df[df["harness"].isin(["markov", "nonmarkov", "reasoner"])].copy()

if hf_df.empty:
    print("No HuggingFace LLM/VLM results found yet. Run some configs first.")
else:
    print(f"HF configs: {hf_df['config'].nunique()}")
    print(f"Models: {sorted(hf_df['model'].unique())}")
    print(f"Harnesses: {sorted(hf_df['harness'].unique())}")
    print(f"Obs modes: {sorted(hf_df['obs_mode'].unique())}")

    # --- 5a. Harness Comparison (grouped bar: harness x model) ---
    harness_agg = hf_df.groupby(["model", "harness"])["success_rate"].mean().unstack("harness")
    harness_order = ["markov", "nonmarkov", "reasoner"]
    harness_agg = harness_agg.reindex(columns=harness_order)
    harness_labels = {"markov": "Markovian ZS", "nonmarkov": "Non-Markovian ZS", "reasoner": "Reasoner (CoT)"}

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(harness_agg))
    w = 0.25
    colors = ["#2196F3", "#FF9800", "#4CAF50"]
    for i, h in enumerate(harness_order):
        if h in harness_agg.columns:
            vals = harness_agg[h].fillna(0)
            bars = ax.bar(x + i * w, vals, w, label=harness_labels.get(h, h), color=colors[i])
            for bar, val in zip(bars, vals):
                if val > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f"{val:.0%}",
                            ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x + w)
    ax.set_xticklabels(harness_agg.index, fontsize=8)
    ax.set_ylabel("Mean Success Rate")
    ax.set_ylim(0, 1.1)
    ax.set_title("Harness Comparison by Model")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- 5b. Observation Mode Comparison ---
    obs_agg = hf_df.groupby(["model", "obs_mode"])["success_rate"].mean().unstack("obs_mode")
    fig, ax = plt.subplots(figsize=(6, 4))
    obs_modes = sorted(hf_df["obs_mode"].unique())
    x = np.arange(len(obs_agg))
    w = 0.35
    obs_colors = {"lang": "#9C27B0", "ascii": "#009688"}
    for i, om in enumerate(obs_modes):
        if om in obs_agg.columns:
            vals = obs_agg[om].fillna(0)
            bars = ax.bar(x + i * w, vals, w, label=f"{om} obs", color=obs_colors.get(om, f"C{i}"))
            for bar, val in zip(bars, vals):
                if val > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f"{val:.0%}",
                            ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x + w / 2)
    ax.set_xticklabels(obs_agg.index, fontsize=8)
    ax.set_ylabel("Mean Success Rate")
    ax.set_ylim(0, 1.1)
    ax.set_title("Observation Mode Comparison by Model")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- 5c. Model Size Scaling ---
    def _get_size(model_name):
        if "30b_a3b" in model_name:
            return "30B-A3B"
        if "4b" in model_name:
            return "4B"
        if "8b" in model_name:
            return "8B"
        return model_name
    def _get_family(model_name):
        if "vl" in model_name:
            return "VLM"
        return "LLM"

    hf_df["size"] = hf_df["model"].apply(_get_size)
    hf_df["family"] = hf_df["model"].apply(_get_family)

    size_agg = hf_df.groupby(["family", "size"])["success_rate"].mean().unstack("size")
    fig, ax = plt.subplots(figsize=(6, 3.5))
    size_agg.plot(kind="bar", ax=ax, color=["#42A5F5", "#1565C0", "#0D47A1"][:len(size_agg.columns)])
    ax.set_ylabel("Mean Success Rate")
    ax.set_ylim(0, 1.1)
    ax.set_title("Model Size Scaling")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title="Params")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- 5d. Difficulty scaling by harness ---
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
    for idx, h in enumerate(harness_order):
        ax = axes[idx]
        sub = hf_df[hf_df["harness"] == h]
        models = sorted(sub["model"].unique())
        palette_h = plt.cm.Set2(np.linspace(0, 1, max(len(models), 2)))
        for mi, model in enumerate(models):
            msub = sub[sub["model"] == model].groupby("difficulty")["success_rate"].mean()
            msub = msub.reindex(DIFF_ORDER)
            ax.plot(DIFF_ORDER, msub, "o-", label=model, color=palette_h[mi], linewidth=1.5, markersize=4)
        ax.set_title(harness_labels.get(h, h), fontsize=10)
        ax.set_ylabel("Success Rate" if idx == 0 else "")
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=6)
        ax.grid(True, alpha=0.3)
    plt.suptitle("Difficulty Scaling by Harness Preset", fontsize=12)
    plt.tight_layout()
    plt.show()

### 5e. PPO vs Best LLM vs Best VLM — Per-Category Bar Comparison

Pick the best config per agent type and compare head-to-head.

In [ ]:
if not df.empty:
    # Find best config per agent type
    type_sr = df.groupby(["config", "agent_type"])["success_rate"].mean().reset_index()
    best_per_type = type_sr.loc[type_sr.groupby("agent_type")["success_rate"].idxmax()]
    representatives = {}
    for _, row in best_per_type.iterrows():
        representatives[row["agent_type"]] = row["config"]

    rep_types = [t for t in ["ppo", "llm", "vlm", "api", "baseline"] if t in representatives]
    if len(rep_types) >= 2:
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(CATEGORY_ORDER))
        w = 0.8 / len(rep_types)
        type_colors = {"ppo": "#E69F00", "llm": "#56B4E9", "vlm": "#009E73", "api": "#D55E00", "baseline": "#999999"}
        for i, at in enumerate(rep_types):
            cfg = representatives[at]
            sub = df[df["config"] == cfg]
            vals = [sub[sub["category"] == c]["success_rate"].mean() if len(sub[sub["category"] == c]) > 0 else 0
                    for c in CATEGORY_ORDER]
            ax.bar(x + i * w, vals, w, label=f"{at}: {cfg}", color=type_colors.get(at, f"C{i}"))
        ax.set_xticks(x + w * (len(rep_types) - 1) / 2)
        ax.set_xticklabels([c.replace("_", "\n") for c in CATEGORY_ORDER], fontsize=7)
        ax.set_ylabel("Mean Success Rate")
        ax.set_ylim(0, 1.1)
        ax.set_title("Best Agent per Type — Per-Category Comparison")
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Only {len(rep_types)} agent type(s) found: {rep_types}")

## 6. Per-Config Heatmaps (Task x Difficulty)

In [ ]:
def plot_heatmap(df_config, config_name, metric="success_rate", cmap="RdYlGn"):
    task_cat = df_config.drop_duplicates("task").set_index("task")["category"]
    cat_rank = {c: i for i, c in enumerate(CATEGORY_ORDER)}
    tasks_sorted = sorted(task_cat.index, key=lambda t: (cat_rank.get(task_cat[t], 99), t))
    pivot = df_config.pivot_table(index="task", columns="difficulty", values=metric, aggfunc="mean")
    pivot = pivot.reindex(index=tasks_sorted, columns=DIFF_ORDER)

    fig, ax = plt.subplots(figsize=(5.5, max(7, len(tasks_sorted) * 0.27)))
    im = ax.imshow(pivot.values, cmap=cmap, aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(DIFF_ORDER)))
    ax.set_xticklabels(DIFF_ORDER)
    ax.set_yticks(range(len(tasks_sorted)))
    ylabels = ax.set_yticklabels(tasks_sorted, fontsize=7)
    for label, task in zip(ylabels, tasks_sorted):
        label.set_color(CAT_COLOR_MAP.get(task_cat.get(task, "unknown"), "gray"))
    for i, task in enumerate(tasks_sorted):
        for j, diff in enumerate(DIFF_ORDER):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=6,
                        color="white" if val < 0.5 else "black")
    plt.colorbar(im, ax=ax, shrink=0.5, label=metric)
    ax.set_title(f"{config_name}", fontsize=10)
    plt.tight_layout()
    plt.show()

if not df.empty:
    for config in configs:
        plot_heatmap(df[df["config"] == config], config)

## 7. Cost & Efficiency

Token usage, latency, and episode length across LLM/VLM agents.

In [ ]:
if not df.empty:
    llm_vlm = df[df["agent_type"].isin(["llm", "vlm", "api"])]
    if not llm_vlm.empty and llm_vlm["mean_latency"].sum() > 0:
        cost_agg = llm_vlm.groupby("config").agg(
            success_rate=("success_rate", "mean"),
            mean_latency=("mean_latency", "mean"),
            total_tokens=("total_tokens", "sum"),
            mean_length=("mean_length", "mean"),
        )

        fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

        # Latency vs success rate
        ax = axes[0]
        for cfg in cost_agg.index:
            at = df[df["config"] == cfg]["agent_type"].iloc[0]
            color = {"llm": "#56B4E9", "vlm": "#009E73", "api": "#D55E00"}.get(at, "gray")
            ax.scatter(cost_agg.loc[cfg, "mean_latency"], cost_agg.loc[cfg, "success_rate"],
                       s=60, color=color, edgecolors="black", linewidths=0.5, zorder=3)
            ax.annotate(cfg, (cost_agg.loc[cfg, "mean_latency"], cost_agg.loc[cfg, "success_rate"]),
                        fontsize=5, xytext=(3, 3), textcoords="offset points")
        ax.set_xlabel("Mean Latency (s/step)")
        ax.set_ylabel("Success Rate")
        ax.set_title("Latency vs Success")
        ax.grid(True, alpha=0.3)

        # Tokens vs success rate
        ax = axes[1]
        for cfg in cost_agg.index:
            at = df[df["config"] == cfg]["agent_type"].iloc[0]
            color = {"llm": "#56B4E9", "vlm": "#009E73", "api": "#D55E00"}.get(at, "gray")
            ax.scatter(cost_agg.loc[cfg, "total_tokens"], cost_agg.loc[cfg, "success_rate"],
                       s=60, color=color, edgecolors="black", linewidths=0.5, zorder=3)
            ax.annotate(cfg, (cost_agg.loc[cfg, "total_tokens"], cost_agg.loc[cfg, "success_rate"]),
                        fontsize=5, xytext=(3, 3), textcoords="offset points")
        ax.set_xlabel("Total Tokens")
        ax.set_ylabel("Success Rate")
        ax.set_title("Token Cost vs Success")
        ax.grid(True, alpha=0.3)

        # Episode length comparison (all agent types)
        ax = axes[2]
        len_agg = df.groupby("config")["mean_length"].mean().sort_values()
        colors = [{"ppo": "#E69F00", "llm": "#56B4E9", "vlm": "#009E73", "api": "#D55E00", "baseline": "#999"}.get(
            df[df["config"] == c]["agent_type"].iloc[0], "gray") for c in len_agg.index]
        ax.barh(range(len(len_agg)), len_agg.values, color=colors)
        ax.set_yticks(range(len(len_agg)))
        ax.set_yticklabels(len_agg.index, fontsize=6)
        ax.set_xlabel("Mean Episode Length")
        ax.set_title("Episode Length by Config")
        ax.grid(axis="x", alpha=0.3)

        plt.tight_layout()
        plt.show()
    else:
        print("No latency/token data available yet.")

## 8. Learning Curves (PPO / RL configs only)

In [ ]:
if not df.empty:
    for config in configs:
        config_curves = {(t, d): v for (c, t, d), v in training_curves.items() if c == config}
        tasks_with_curves = sorted(set(t for t, d in config_curves.keys()))
        if not tasks_with_curves:
            continue

        n = len(tasks_with_curves)
        ncols = 6
        nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.2), squeeze=False)

        for idx, task in enumerate(tasks_with_curves):
            ax = axes[idx // ncols][idx % ncols]
            for diff in DIFF_ORDER:
                curve = config_curves.get((task, diff), [])
                if not curve:
                    continue
                steps = [p["timestep"] / 1000 for p in curve]
                rewards = [p["mean_reward"] for p in curve]
                stds = [p.get("std_reward", 0) for p in curve]
                ax.plot(steps, rewards, label=diff, color=DIFF_COLORS[diff], linewidth=1)
                ax.fill_between(steps, np.array(rewards) - np.array(stds),
                                np.array(rewards) + np.array(stds), alpha=0.1, color=DIFF_COLORS[diff])
            ax.set_title(task.replace("-v0", ""), fontsize=7)
            ax.tick_params(labelsize=6)
            ax.set_xlabel("steps (k)", fontsize=6)

        for idx in range(n, nrows * ncols):
            axes[idx // ncols][idx % ncols].set_visible(False)

        handles, labels = axes[0][0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper right", fontsize=8)
        fig.suptitle(f"{config} — Learning Curves", fontsize=12)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

## 9. Failure Analysis

In [ ]:
if not df.empty:
    for config in configs:
        sub = df[df["config"] == config]
        header_printed = False

        # Errored tasks
        errored = sub[sub["error"].notna()]
        if len(errored) > 0:
            if not header_printed:
                print(f"\n{'=' * 60}\n{config}\n{'=' * 60}")
                header_printed = True
            print(f"  ERRORED ({len(errored)}):")
            for _, row in errored.head(10).iterrows():
                print(f"    {row['task']:30s} {row['difficulty']:8s}  {str(row['error'])[:60]}")

        # Zero success on easy
        easy = sub[(sub["difficulty"] == "easy") & (sub["success_rate"] == 0.0) & (sub["error"].isna())]
        if len(easy) > 0:
            if not header_printed:
                print(f"\n{'=' * 60}\n{config}\n{'=' * 60}")
                header_printed = True
            print(f"  ZERO SUCCESS ON EASY ({len(easy)} tasks):")
            for _, row in easy.iterrows():
                print(f"    {row['task']:30s}  return={row['mean_return']:.3f}")

        # Perfect on all difficulties
        perfect = sub.groupby("task").filter(lambda g: (g["success_rate"] == 1.0).all())
        if len(perfect) > 0:
            tasks = sorted(perfect["task"].unique())
            if not header_printed:
                print(f"\n{'=' * 60}\n{config}\n{'=' * 60}")
                header_printed = True
            print(f"  PERFECT ON ALL DIFFICULTIES ({len(tasks)} tasks): {', '.join(tasks)}")